In [1]:
from pathlib import Path

import pandas as pd

csv_path = Path("../data/processed/cases_v5_full.csv")
df = pd.read_csv(csv_path)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Missing mode_of_expression: {df['mode_of_expression'].isna().sum():,}")
print(f"Unique exact values: {df['mode_of_expression'].nunique(dropna=True):,}")

Rows: 1,075
Columns: 250
Missing mode_of_expression: 9
Unique exact values: 51


In [2]:
mode_categories = (
    df["mode_of_expression"]
    .dropna()
    .astype(str)
    .str.replace(r"\s*\|\s*", ", ", regex=True)
    .str.split(r"\s*,\s*", regex=True)
    .explode()
    .str.strip()
)

category_counts = (
    mode_categories.value_counts()
    .rename_axis("mode_of_expression_category")
    .reset_index(name="count")
)

category_counts

,mode_of_expression_category,count
0,Electronic / Internet-based Communication,381
1,Press / Newspapers,227
2,Non-verbal Expression,120
3,Public Speech,104
4,Public Documents,103
5,Audio / Visual Broadcasting,88
6,Public Assembly,56
7,Pamphlets / Posters / Banners,38
8,Books / Plays,34
9,Written speech,28


In [3]:
exact_value_counts = (
    df["mode_of_expression"]
    .fillna("Missing")
    .value_counts()
    .rename_axis("mode_of_expression")
    .reset_index(name="count")
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
):
    display(exact_value_counts)

,mode_of_expression,count
0,Electronic / Internet-based Communication,345
1,Press / Newspapers,190
2,Public Documents,102
3,Non-verbal Expression,102
4,Public Speech,71
5,Audio / Visual Broadcasting,61
6,Public Assembly,41
7,Books / Plays,27
8,Pamphlets / Posters / Banners,24
9,Written speech,11


In [4]:
cases_with_mode_dummies = df.copy()

mode_dummy_groups = {
    "mode_electronic_internet": ["Electronic / Internet-based Communication"],
    "mode_press_newspapers": ["Press / Newspapers"],
    "mode_public_speech": ["Public Speech"],
    "mode_public_documents": ["Public Documents", "Public Documents/Information"],
    "mode_audio_visual_broadcasting": ["Audio / Visual Broadcasting"],
    "mode_public_assembly": ["Public Assembly"],
    "mode_written_speech": ["Written speech", "Books / Plays", "Pamphlets / Posters / Banners"],
    "mode_non_verbal_expression": ["Non-verbal Expression"],
}

mode_parts_by_row = (
    df["mode_of_expression"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s*\|\s*", ", ", regex=True)
    .str.split(r"\s*,\s*", regex=True)
    .apply(lambda values: {value.strip() for value in values if value.strip()})
)

for column, labels in mode_dummy_groups.items():
    label_set = set(labels)
    cases_with_mode_dummies[column] = mode_parts_by_row.apply(
        lambda values: int(bool(values & label_set))
    )

mode_dummy_columns = list(mode_dummy_groups)

mode_dummy_counts = (
    cases_with_mode_dummies[mode_dummy_columns]
    .sum()
    .rename_axis("dummy_variable")
    .reset_index(name="count")
)

mode_dummy_counts

,dummy_variable,count
0,mode_electronic_internet,381
1,mode_press_newspapers,227
2,mode_public_speech,104
3,mode_public_documents,106
4,mode_audio_visual_broadcasting,88
5,mode_public_assembly,56
6,mode_written_speech,95
7,mode_non_verbal_expression,120


In [5]:
output_path = Path("../data/processed/cases_v5_full_modes.csv")
cases_with_mode_dummies.to_csv(output_path, index=False)

print(
    f"Saved {output_path} with "
    f"{cases_with_mode_dummies.shape[0]:,} rows and "
    f"{cases_with_mode_dummies.shape[1]:,} columns."
)

Saved ../data/processed/cases_v5_full_modes.csv with 1,075 rows and 258 columns.


In [6]:
short_path = Path("../data/processed/cases_v5_short.csv")
short_df = pd.read_csv(short_path)

mode_expression_matches = df["mode_of_expression"].fillna("__MISSING__").equals(
    short_df["mode_of_expression"].fillna("__MISSING__")
)

print(f"Short rows: {len(short_df):,}")
print(f"Short columns before dummies: {short_df.shape[1]:,}")
print(f"mode_of_expression matches full file: {mode_expression_matches}")

cases_short_with_mode_dummies = short_df.copy()

short_mode_parts_by_row = (
    short_df["mode_of_expression"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s*\|\s*", ", ", regex=True)
    .str.split(r"\s*,\s*", regex=True)
    .apply(lambda values: {value.strip() for value in values if value.strip()})
)

for column, labels in mode_dummy_groups.items():
    label_set = set(labels)
    cases_short_with_mode_dummies[column] = short_mode_parts_by_row.apply(
        lambda values: int(bool(values & label_set))
    )

short_mode_dummy_counts = (
    cases_short_with_mode_dummies[mode_dummy_columns]
    .sum()
    .rename_axis("dummy_variable")
    .reset_index(name="short_count")
)

mode_count_verification = mode_dummy_counts.merge(
    short_mode_dummy_counts,
    on="dummy_variable",
    how="outer",
)
mode_count_verification["matches"] = (
    mode_count_verification["count"] == mode_count_verification["short_count"]
)

assert mode_expression_matches
assert mode_count_verification["matches"].all()

mode_count_verification

Short rows: 1,075
Short columns before dummies: 114
mode_of_expression matches full file: True


,dummy_variable,count,short_count,matches
0,mode_audio_visual_broadcasting,88,88,True
1,mode_electronic_internet,381,381,True
2,mode_non_verbal_expression,120,120,True
3,mode_press_newspapers,227,227,True
4,mode_public_assembly,56,56,True
5,mode_public_documents,106,106,True
6,mode_public_speech,104,104,True
7,mode_written_speech,95,95,True


In [7]:
short_output_path = Path("../data/processed/cases_v5_short_modes.csv")
cases_short_with_mode_dummies.to_csv(short_output_path, index=False)

print(
    f"Saved {short_output_path} with "
    f"{cases_short_with_mode_dummies.shape[0]:,} rows and "
    f"{cases_short_with_mode_dummies.shape[1]:,} columns."
)

Saved ../data/processed/cases_v5_short_modes.csv with 1,075 rows and 122 columns.
